## Gold Layer — Business Aggregations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import when

silver_df = spark.table("healthflow_catalog.silver.patients_clean")

# ── Conditions Analysis ──────────────────────────────────────────────────────
conditions = silver_df.groupBy("medical_condition").agg(
    F.count("*").alias("patient_count"),
    F.round(F.avg("billing_amount"), 2).alias("avg_billing"),
    F.round(F.avg("length_of_stay"), 1).alias("avg_stay_days"),
)
conditions.write.format("delta").mode("overwrite").saveAsTable(
    "healthflow_catalog.gold.conditions_analysis"
)

# ── Hospital Performance ─────────────────────────────────────────────────────
hospitals = silver_df.groupBy("hospital").agg(
    F.count("*").alias("total_patients"),
    F.round(F.avg("billing_amount"), 2).alias("avg_billing"),
    F.sum(when(F.col("admission_type") == "Urgent", 1).otherwise(0)).alias("urgent_count"),
)
hospitals.write.format("delta").mode("overwrite").saveAsTable(
    "healthflow_catalog.gold.hospital_performance"
)

# ── Demographics ─────────────────────────────────────────────────────────────
demographics = (
    silver_df
    .withColumn(
        "age_group",
        when(F.col("age") <= 18, "0-18")
        .when(F.col("age") <= 35, "19-35")
        .when(F.col("age") <= 50, "36-50")
        .when(F.col("age") <= 65, "51-65")
        .otherwise("65+"),
    )
    .groupBy("age_group")
    .agg(
        F.count("*").alias("patient_count"),
        F.round(F.avg("billing_amount"), 2).alias("avg_billing"),
    )
)
demographics.write.format("delta").mode("overwrite").saveAsTable(
    "healthflow_catalog.gold.demographics"
)

print("Gold layer complete — conditions, hospitals, demographics written")

Gold layer complete — conditions, hospitals, demographics written
